## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

import glob
import os
from pathlib import Path

import pickle

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
json_files_dict = {}

for path in sorted(Path('../raw_data-originals/dati_eumaco_con_pseudo').glob('*.json')):
    try:
        filename = path.stem
        json_content = json.loads(path.read_text()) # Closes automatically the file
        json_files_dict[filename] = json_content
    except json.JSONDecodeError:
        print(f"Skipping invalid JSON: {path}")

# List of all the json files names
json_filenames = list(json_files_dict)

## Make a pretty-print version of all the jsons

In [ ]:
# def pretty_print_folder(input_dir, output_dir):
#     os.makedirs(output_dir, exist_ok=True)

#     for filename in os.listdir(input_dir):
#         with open(os.path.join(input_dir, filename), "r", encoding="utf-8") as file:
#             data = json.load(file)

#         name = filename[:-5]  # strip ".json"
#         out_path = os.path.join(output_dir, f"{name}-pp.json")

#         with open(out_path, "w", encoding="utf-8") as file:
#             json.dump(data, file, indent=2, ensure_ascii=False, sort_keys=False)

In [ ]:
# input_dir = '../raw_data-originals/dati_eumaco_con_pseudo'
# output_dir = '../raw_data/jsons-pp'

# pretty_print_folder(input_dir, output_dir)

## Analysis and overview

### Check if the "external" fields except the `transmissions` have the same structure

In [ ]:
# def compare_general_jsons(json_files_dict, json_filenames, sample_json_name, *, names=True):
#     '''
#     Takes in input:
#     - a list of json filenames
#     - a dictionary {json_filename: json_file_content}
#     - a sample instance to check against
#     - the json_name option

#     Returns an overview list that tells the name of the json files that have a structure different from the sample instance
#     '''

#     different_general_jsons = []
#     sample_json = show_structure_simple(json_files_dict[sample_json_name])
#     sample_json.pop('transmissions')

#     # Iterates over the jsons
#     for json_name in json_filenames:
#         current_json = show_structure_simple(json_files_dict[json_name])
#         current_json.pop('transmissions')

#         if sample_json != current_json:

#             if names == True:
#                 different_general_jsons.append(json_name)
#             else:
#                 different_general_jsons.append(current_json)

#     return different_general_jsons

In [ ]:
# sample_json = show_structure_simple(json_files_dict['31636'])
# sample_json

## Parsing and reformatting

Shows a sample of the field where the useful data is stored 

In [ ]:
json_files_dict[json_filenames[0]]['transmissions']['SpO2Transmission'][0][0]

### Put specific values of the json in a pandas df

In [ ]:
def get_deep(dic, path):
    '''
    Helper function that takes in input:
    - a dictionary
    - a path to a value of the dictionary, colon separated
    and returns the corresponding value, even if the key:value pair is nested inside the dictionary

    For example, running the function on `d = {'ext': {'int': 'val'}}`:
    - get_deep(d, 'ext') -> {'int': 'val'}
    - get_deep(d, 'ext:int') -> 'val'
    '''
    
    current = dic
    for key in path.split(':'):
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return None
    return current

In [ ]:
def json_to_df(json_files_dict, json_filenames, *, just_external=False, transm_field=None, external_json_fields= None, transm_inst_fields=None):
    '''
    Takes in input:
    - a dictionary {json_filename: json_file_content}
    - a list of json filenames
    - the just_external bool option
    - the transmission subfield to check
    - the external fields common to every json file
    - the fields internal to the transmission subfield to check

    Iterates over the json files and the transmission field instances

    Returns a list containing dictionaries, where each one carries the informations
    of every instance of a specific transmission field,
    merged with the informations of the json itself

    If just_external is True, returns just the external informations of the json
    (number of rows will be equal to the number of json files)
    '''

    data = []

    # Iterates over the jsons
    for json_name in json_filenames:
        
        # current_json is 
        current_json = json_files_dict[json_name]

        # Dictionary containing all the "external" information of a json,
        # common to every instance of a transmission
        helper_dict = {}

        # Iterates over a list of external fields
        for field in external_json_fields:

            # Gets the value of the required field ...
            value = get_deep(current_json, field)
            # and puts it into the helper_dict
            helper_dict[field] = value

        # Return only the fields external to the transmission field
        if just_external == True:
            # Appends the data to the initial list
            data.append(helper_dict)
            continue

        # TODO: Maybe is better to place NAs?
        if transm_field not in current_json['transmissions']:
            continue

        # list containing all the instances of a transmission field
        instances = current_json['transmissions'][transm_field][0]

        for instance in instances:
            # Makes a copy of the common dictionary,
            # to allow to replace it at every loop
            instance_helper_dict = helper_dict
            
            # Iterates over a list of internal transmission fields
            for field in transm_inst_fields:

               # Gets the value of the required field ... 
                value = get_deep(instance, field)
                # and puts it into the instance_helper_dict
                instance_helper_dict[field] = value

                # Appends the data to the initial list
                data.append(instance_helper_dict)

    return data

In [ ]:
external_fields = [
    'patient_cf' ,
    'assignment_id' ,
    'archivied' ,
    'completed' ,

    'assignment:id' ,
    'assignment:start_at' ,
    'assignment:end_at' ,
    'assignment:active' ,
    'assignment:created_at' ,
    'assignment:updated_at' ,
    'assignment:hub_sn' ,
    'assignment:completed' ,
]

instance_ext_fields = [
    'id' ,
    'type' ,
    'report' ,
    'created_at' ,
    'updated_at' ,
    'registration_time' ,
    'device_sn' ,

# transmission_data
    'transmission_data:device' ,
    'transmission_data:serialnumber' ,
    'transmission_data:timestamp' ,
# battery
    'transmission_data:battery:value' ,
    'transmission_data:battery:unit' ,
# spo2
    'transmission_data:spo2:value' ,
    'transmission_data:spo2:unit' ,
# heartrate
    'transmission_data:heartrate:value' ,
    'transmission_data:heartrate:unit' ,
]

In [ ]:
data = json_to_df(json_files_dict, json_filenames,
    just_external = True,
    transm_field = 'SpO2Transmission',
    external_json_fields = external_fields,
    transm_inst_fields = instance_ext_fields
)
df = pd.DataFrame(data)

#### Old version (to be deleted)

In [ ]:
# fields = {
# #  <df-column_name> : <json-field_path>  
#     "patient_cf": "patient_cf",
#     # "device_sn": "assignment.hub_sn",
#     "archivied": "archivied",
#     "completed": "completed",

#     # examples of nested fields:
#     # "device_sn": "transmissions.SpO2Transmission.0.0.device_sn",
#     # "spo2": "transmissions.SpO2Transmission.0.0.transmission_data.spo2.value",
# }


# def get_nested(data, field_path, default=None):

#     current = data
#     try:
#         # Loops over the json tree
#         for key in field_path.split('.'):
#             if key.isdigit():
#                 key = int(key)
#             current = current[key]
#         return current

#     except (KeyError, IndexError, TypeError):
#         return default

# helper_dic = {}
# for column, field in fields.items():
#     helper_dic[column] = []
#     for json in json_files_list:
#         helper_dic[column].append(get_nested(json, field))

# df = pd.DataFrame(helper_dic)
# df['archivied'].value_counts(dropna=False).to_dict() # Unsorted dict
# df

### Put all the `patient_cf` fields in a list

In [ ]:
# ids = []

# for json in json_files:
#     ids.append(json['patient_cf'])

In [ ]:
# ser = pd.Series(ids)
# ser

## Export

In [ ]:
# # Pickle
# with open('../raw_data/pickles/4-eumaco.pickle', 'wb') as file:
#     pickle.dump(df, file, protocol=pickle.HIGHEST_PROTOCOL)